Basic setup – imports and file paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# -------------------------------------------------------------------
# Adjust these paths to where your CSV files are saved
# -------------------------------------------------------------------
DATA_DIR = Path("/Users/imenhassine/Documents/MASTER/3rd Semester/QARM II/Project/QARM-II/Data")  # e.g. Path("C:/Users/you/Desktop")

FILE_INDICES = DATA_DIR / "Indices global 01-2005 à 10-2025.csv"
FILE_STOCKS  = DATA_DIR / "Stocks global 01-2005 à 10-2025.csv"

Step 1 – Load CSVs and inspect structure

In [2]:
# -------------------------------------------------------------------
# Load the raw CSVs
# -------------------------------------------------------------------
indices_raw = pd.read_csv(FILE_INDICES, encoding="latin-1")
stocks_raw  = pd.read_csv(FILE_STOCKS,  encoding="latin-1")

# -------------------------------------------------------------------
# Inspect columns
# -------------------------------------------------------------------
print("=== INDICES COLUMNS ===")
print(indices_raw.columns.tolist())
print("\nIndices sample:")
print(indices_raw.head())

print("\n=== STOCKS COLUMNS ===")
print(stocks_raw.columns.tolist())
print("\nStocks sample:")
print(stocks_raw.head())


/var/folders/p3/blqhp3j513n1jckp7yphf8q00000gn/T/ipykernel_41695/4167715820.py:5: DtypeWarning: Columns (0,3) have mixed types. Specify dtype option on import or set low_memory=False.
  stocks_raw  = pd.read_csv(FILE_STOCKS,  encoding="latin-1")


=== INDICES COLUMNS ===
['gvkeyx', 'datadate', 'indextype', 'tic', 'indexid', 'indexcat', 'idxiddesc', 'prccm']

Indices sample:
   gvkeyx    datadate  indextype       tic indexid indexcat  \
0  115114  2005-01-31  COMPOSITE  I2JPN017  JASDAQ    EXCHG   
1  115114  2005-02-28  COMPOSITE  I2JPN017  JASDAQ    EXCHG   
2  115114  2005-03-31  COMPOSITE  I2JPN017  JASDAQ    EXCHG   
3  115114  2005-04-30  COMPOSITE  I2JPN017  JASDAQ    EXCHG   
4  115114  2005-05-31  COMPOSITE  I2JPN017  JASDAQ    EXCHG   

                            idxiddesc  prccm  
0  Japanese Over the Counter Exchange  95.69  
1  Japanese Over the Counter Exchange  95.49  
2  Japanese Over the Counter Exchange  95.68  
3  Japanese Over the Counter Exchange  95.58  
4  Japanese Over the Counter Exchange  93.24  

=== STOCKS COLUMNS ===
['fic', 'gvkey', 'datadate', 'conm', 'ajpm', 'prccm', 'curcddvm', 'dvpspm', 'exchg']

Stocks sample:
   fic  gvkey    datadate conm  ajpm     prccm curcddvm  dvpspm  exchg
0  NaN      5 

Helper: function to compute returns from prices

In [3]:
def compute_log_returns(df, date_col, id_col, price_col):
    """
    df: long dataframe (date, asset_id, price)
    date_col: column name with dates
    id_col: column name with asset identifiers
    price_col: column name with prices (close)
    """
    df = df.copy()
    
    # Ensure dates are datetime and sorted
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values([id_col, date_col])
    
    # Compute log returns by asset
    df["log_ret"] = (
        np.log(df.groupby(id_col)[price_col].shift(-1)) 
        - np.log(df[price_col])
    )
    
    # Drop last observation per asset (no forward return)
    df = df.dropna(subset=["log_ret"])
    
    return df


Step 2 – Clean & transform the indices file

In [4]:
# -------------------------------------------------------------------
# Clean indices data
# -------------------------------------------------------------------

# Choose relevant columns (modify if your names differ)
indices_cols = ["datadate", "tic", "prccm"]
indices = indices_raw[indices_cols].copy()

# Rename for clarity
indices = indices.rename(columns={
    "datadate": "date",
    "tic": "asset_id",
    "prccm": "price"
})

# Compute log returns
indices_ret_long = compute_log_returns(
    df=indices,
    date_col="date",
    id_col="asset_id",
    price_col="price"
)

# Pivot to wide matrix: rows = dates, columns = indices, values = log returns
indices_ret_wide = indices_ret_long.pivot(index="date", columns="asset_id", values="log_ret")

# Optional: sort index
indices_ret_wide = indices_ret_wide.sort_index()

print("Indices returns shape:", indices_ret_wide.shape)
print(indices_ret_wide.head())


Indices returns shape: (249, 735)
asset_id    86UNK080  I1EGY001  I1GHA001  I1ISR001  I1ISR002  I1JOR001  \
date                                                                     
2005-01-31  0.152000  0.116466 -0.022344  0.021011  0.017429  0.009802   
2005-02-28 -0.120310 -0.032452 -0.042971 -0.011563 -0.006082  0.092810   
2005-03-31 -0.094199  0.061844 -0.055045  0.017983  0.022529  0.211473   
2005-04-30  0.044247  0.025143 -0.009567  0.032852  0.034076 -0.023108   
2005-05-31  0.064793  0.116617 -0.031446 -0.063188 -0.061863  0.102336   

asset_id    I1KEN001  I1MAR001  I1NGA001  I1QAT001  ...  I6UNK170  I6UNK171  \
date                                                ...                       
2005-01-31  0.037558 -0.025755 -0.049771  0.258286  ...  0.028086  0.079204   
2005-02-28 -0.027369 -0.013714 -0.059645  0.162503  ...  0.005333 -0.019070   
2005-03-31  0.031959  0.020104  0.060018 -0.041254  ... -0.064987 -0.026078   
2005-04-30  0.082566  0.046338 -0.022081 -0.112156  

Step 3 – Clean & transform the stocks file

In [7]:
# -------------------------------------------------------------------
# Clean stocks data
# -------------------------------------------------------------------

# Choose relevant columns (modify names if needed).
# Some extracts include 'iid' (issue id) and some do not.
# Use a safe selection depending on available columns.
if "iid" in stocks_raw.columns:
    stocks_cols = ["datadate", "gvkey", "iid", "prccm"]
else:
    stocks_cols = ["datadate", "gvkey", "prccm"]

stocks = stocks_raw[stocks_cols].copy()

# Build a unique asset identifier. If iid exists use gvkey-iid, otherwise gvkey alone.
if "iid" in stocks.columns:
    stocks["asset_id"] = stocks["gvkey"].astype(str) + "-" + stocks["iid"].astype(str)
else:
    stocks["asset_id"] = stocks["gvkey"].astype(str)

stocks = stocks.rename(columns={
    "datadate": "date",
    "prccm": "price"
})

stocks = stocks[["date", "asset_id", "price"]]

# Compute log returns
stocks_ret_long = compute_log_returns(
    df=stocks,
    date_col="date",
    id_col="asset_id",
    price_col="price"
)

# Pivot to wide matrix
# There can be duplicate (date, asset_id) rows which cause pivot to fail.
# Aggregate duplicates (e.g. by mean) before reshaping.
stocks_ret_agg = (
    stocks_ret_long
    .groupby(["date", "asset_id"], as_index=False)["log_ret"]
    .mean()
)

stocks_ret_wide = stocks_ret_agg.pivot(index="date", columns="asset_id", values="log_ret")
stocks_ret_wide = stocks_ret_wide.sort_index()

print("Stocks returns shape:", stocks_ret_wide.shape)
print(stocks_ret_wide.head())


/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Stocks returns shape: (238, 57860)
asset_id      100001    100003    100004    100010    100012    100013  \
date                                                                     
2006-01-31  0.011225 -0.083382 -0.019772  0.040103  0.035156 -0.017293   
2006-02-28 -0.029390  0.122602  0.126350 -0.020665  0.018019  0.039909   
2006-03-31  0.023809  0.143101 -0.005294  0.202478  0.007117  0.002700   
2006-04-30 -0.043205 -0.105361 -0.096727 -0.062977 -0.128478 -0.031970   
2006-05-31  0.019606 -0.076961 -0.058408  0.018870 -0.004040 -0.016044   

asset_id      100021    100022    100023    100025  ...      8169      8303  \
date                                                ...                       
2006-01-31  0.023605  0.040114 -0.018692  0.032039  ... -0.016832 -0.364404   
2006-02-28  0.014666  0.059635  0.006270  0.013423  ... -0.014078  0.000000   
2006-03-31  0.027958 -0.026429  0.192684 -0.050124  ...  0.010577  0.144250   
2006-04-30  0.023565 -0.035691 -0.058372 -0.072671 

Step 4 – Combine into a single multi-asset universe (optional)

In [8]:
# Align dates and concatenate columns
common_dates = indices_ret_wide.index.intersection(stocks_ret_wide.index)

multiasset_ret = pd.concat(
    [
        indices_ret_wide.loc[common_dates],
        stocks_ret_wide.loc[common_dates]
    ],
    axis=1
)

print("Multi-asset return matrix shape:", multiasset_ret.shape)
print(multiasset_ret.head())


Multi-asset return matrix shape: (237, 58595)
asset_id    86UNK080  I1EGY001  I1GHA001  I1ISR001  I1ISR002  I1JOR001  \
date                                                                     
2006-01-31  0.042366 -0.083322  0.007923 -0.025427 -0.029162 -0.092485   
2006-02-28 -0.072036 -0.071837  0.007141  0.015154  0.017252 -0.066458   
2006-03-31  0.076187 -0.024797  0.003372  0.051466  0.056100 -0.004278   
2006-04-30 -0.206216 -0.173879  0.013226 -0.012637 -0.010076 -0.017316   
2006-05-31 -0.051347 -0.095593 -0.002164 -0.090652 -0.088866 -0.133334   

asset_id    I1KEN001  I1MAR001  I1NGA001  I1QAT001  ...      8169      8303  \
date                                                ...                       
2006-01-31 -0.027995  0.045030  0.006883       NaN  ... -0.016832 -0.364404   
2006-02-28  0.011034  0.054221 -0.021467       NaN  ... -0.014078  0.000000   
2006-03-31 -0.018810  0.094296 -0.001517       NaN  ...  0.010577  0.144250   
2006-04-30  0.077541 -0.122189  0.060144